# Interactive Image Processing Application

In this lab, you will build an interactive image processing application using OpenCV, Matplotlib, and ipywidgets. The application will allow you to perform various operations on an image such as converting it to grayscale, adjusting its brightness, resizing, rotating, and plotting its histogram. You will also be able to save the processed image.

In [1]:
# Import necessary libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import os

# Create outputs directory
output_path = './outputs'
os.makedirs(output_path, exist_ok=True)

# Global variables
original_bgr = None
original_rgb = None
processed_rgb = None

# Output widgets
msg_out = widgets.Output()
img_out = widgets.Output()
hist_out = widgets.Output()

# Function to display images
def show_images(original_rgb, processed_rgb):
    with img_out:
        img_out.clear_output()
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(original_rgb)
        axes[0].set_title('Original Image')
        axes[0].axis('off')
        axes[1].imshow(processed_rgb)
        axes[1].set_title('Processed Image')
        axes[1].axis('off')
        plt.show()

# Function to display histogram
def show_histogram(processed_rgb):
    with hist_out:
        hist_out.clear_output()
        gray = cv2.cvtColor(processed_rgb, cv2.COLOR_RGB2GRAY)
        plt.figure(figsize=(5, 3))
        plt.hist(gray.ravel(), bins=256, range=[0, 256], color='gray')
        plt.title('Histogram')
        plt.show()

# Callback functions
def on_browse_click(change):
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded.keys():
        global original_bgr, original_rgb, processed_rgb
        original_bgr = cv2.imread(fname)
        if original_bgr is not None:
            original_rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
            processed_rgb = original_rgb.copy()
            show_images(original_rgb, processed_rgb)
            with msg_out:
                msg_out.clear_output()
                print(f'Image loaded: {fname}')
        else:
            with msg_out:
                msg_out.clear_output()
                print('Failed to load image.')

def on_grayscale_click(change):
    global processed_rgb
    gray = cv2.cvtColor(original_rgb, cv2.COLOR_RGB2GRAY)
    processed_rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    show_images(original_rgb, processed_rgb)

def on_brightness_change(change):
    global processed_rgb
    value = brightness_slider.value
    processed_rgb = cv2.convertScaleAbs(original_rgb, alpha=1, beta=value)
    show_images(original_rgb, processed_rgb)

def on_resize_change(change):
    global processed_rgb
    scale = resize_dropdown.value
    width = int(original_rgb.shape[1] * scale)
    height = int(original_rgb.shape[0] * scale)
    processed_rgb = cv2.resize(original_rgb, (width, height), interpolation=cv2.INTER_AREA)
    show_images(original_rgb, processed_rgb)
    with msg_out:
        msg_out.clear_output()
        print(f'Resized to: {processed_rgb.shape[1]}x{processed_rgb.shape[0]}')

def on_rotation_change(change):
    global processed_rgb
    rotation_map = {0: cv2.ROTATE_180, 90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180, 270: cv2.ROTATE_90_COUNTERCLOCKWISE}
    angle = rotation_dropdown.value
    if angle in rotation_map:
        processed_rgb = cv2.rotate(original_rgb, rotation_map[angle])
        show_images(original_rgb, processed_rgb)

def on_histogram_click(change):
    show_histogram(processed_rgb)

def on_save_click(change):
    global processed_rgb
    save_name = os.path.join(output_path, 'processed_image.png')
    cv2.imwrite(save_name, cv2.cvtColor(processed_rgb, cv2.COLOR_RGB2BGR))
    with msg_out:
        msg_out.clear_output()
        print(f'Image saved as: {save_name}')

# Browse Image button
browse_button = widgets.Button(description='Browse Image')
browse_button.on_click(on_browse_click)

# Grayscale button
grayscale_button = widgets.Button(description='Convert to Grayscale')
grayscale_button.on_click(on_grayscale_click)

# Brightness slider
brightness_slider = widgets.IntSlider(description='Brightness', min=-100, max=100, step=1, value=0)
brightness_slider.observe(on_brightness_change, names='value')

# Resize dropdown
resize_dropdown = widgets.Dropdown(description='Resize', options=[1.0, 0.75, 0.5], value=1.0)
resize_dropdown.observe(on_resize_change, names='value')

# Rotation dropdown
rotation_dropdown = widgets.Dropdown(description='Rotation', options=[0, 90, 180, 270], value=0)
rotation_dropdown.observe(on_rotation_change, names='value')

# Plot Histogram button
histogram_button = widgets.Button(description='Plot Histogram')
histogram_button.on_click(on_histogram_click)

# Save Image button
save_button = widgets.Button(description='Save Image')
save_button.on_click(on_save_click)

# Layout
controls = widgets.VBox([browse_button, grayscale_button, brightness_slider, resize_dropdown, rotation_dropdown, histogram_button, save_button])
images = widgets.VBox([img_out, hist_out])
app_layout = widgets.HBox([controls, images])

# Display the app
display(msg_out, app_layout)
print('Please click on Browse Image to start.')
